# AI-Driven Tourism Recommendation System
## DTS114TC — Software Component

This notebook uses LLM to generate a complete tourism recommendation web application.
All code, diagrams, and documentation are automatically generated following the AI-Driven SDLC methodology.

In [5]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

# Check dependencies
try:
    from PIL import Image
    print("Pillow OK")
except Exception:
    print("Installing Pillow...")
    import subprocess; subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pillow'])

from utils import load_environment, get_completion, setup_llm_client
from utils import clean_llm_output, recommended_models_table, save_artifact, load_artifact
from utils import render_plantuml_diagram, get_image_generation_completion
from IPython.display import display, Markdown, Image as IPyImage

print("All imports loaded.")

Pillow OK
All imports loaded.


In [6]:
load_environment()

MODEL = "openai/gpt-5.2"
IMAGE_MODEL = "qwen/qwen-image-2512"

client, model_name, provider = setup_llm_client(MODEL)
print(f"Provider: {provider}, Model: {model_name}")

# Show available text models
try:
    tbl = recommended_models_table(task="text", min_context=100_000)
except Exception as e:
    print(f"Model table skipped: {e}")

✅ LLM Client configured: Using 'apifree' with model 'openai/gpt-5.2'
Provider: apifree, Model: openai/gpt-5.2


| Model | Provider | Text | Vision | Image Gen | Image Edit | Audio Transcription | Context Window | Max Output Tokens |
|---|---|---|---|---|---|---|---|---|
| deepseek-ai/DeepSeek-V3.1 | huggingface | ✅ | ❌ | ❌ | ❌ | ❌ | 128,000 | 100,000 |
| deepseek-chat | deepseek | ✅ | ❌ | ❌ | ❌ | ❌ | 4,000,000 | 8,000 |
| meta-llama/Llama-4-Maverick-17B-128E-Instruct | huggingface | ✅ | ❌ | ❌ | ❌ | ❌ | 1,000,000 | 100,000 |
| meta-llama/Llama-4-Scout-17B-16E-Instruct | huggingface | ✅ | ❌ | ❌ | ❌ | ❌ | 10,000,000 | 100,000 |

## Phase 1: Inception
Define business intent and generate SDLC documentation.

In [8]:
# Business problem definition
business_problem = (
    "Our company needs a tourism recommendation platform where users can input a city name "
    "and get a curated list of famous attractions along with a personalized travel itinerary. "
    "The system should display AI-generated images of each attraction and export a day-by-day plan."
)

#### Generate Problem Statement

In [9]:
prompt = f"""Turn this business problem into a single, clear problem statement (2-3 sentences max).
Business Problem: {business_problem}"""
problem_statement = get_completion(prompt, client, model_name, provider, temperature=0.3)
print(problem_statement)

SyntaxError: unterminated string literal (detected at line 2) (2868074500.py, line 2)

#### Generate User Personas

In [ ]:
prompt = f"""Generate 4 user personas for a tourism recommendation platform. Use this format:

1. **Role Title**
   - Responsibilities: ...
   - Needs: ...

Rules: single role titles only, no combined titles (no slashes). 2-3 bullets per role.
Problem Statement: {problem_statement}"""
personas = get_completion(prompt, client, model_name, provider, temperature=0.3)
print(personas)

#### Generate Product Requirements Document

In [ ]:
prompt = f"""Write a PRD in markdown with these headings, 2-4 concise bullets each:
## Overview
## Goals
## Non-Goals
## User Personas (brief)
## Key Features
## User Flows
## Functional Requirements
## Non-Functional Requirements
## Constraints/Assumptions
## Success Metrics

Rules: only the headings above, no extra sections. Keep bullets short.
Problem Statement: {problem_statement}
Personas: {personas}"""
prd = get_completion(prompt, client, model_name, provider, temperature=0.3)
prd = clean_llm_output(prd, language='markdown')

os.makedirs('artifacts', exist_ok=True)
with open('artifacts/prd.md', 'w', encoding='utf-8') as f:
    f.write(prd)
print("Saved: artifacts/prd.md")
print(prd[:500])

#### Generate User Stories

In [ ]:
prompt = f"""Return ONLY valid JSON following this schema exactly:
{{
  "user_stories": [
    {{
      "id": 1,
      "role": "<role>",
      "goal": "<goal>",
      "benefit": "<benefit>",
      "acceptance_criteria": ["<criteria>", "<criteria>"]
    }}
  ]
}}

Rules: 5 stories total. Keep each field concise. No extra keys. Role names must be singular.
PRD: {prd}"""
user_stories = get_completion(prompt, client, model_name, provider, temperature=0.3)
user_stories = clean_llm_output(user_stories, language='json')

import json
os.makedirs('artifacts', exist_ok=True)
with open('artifacts/user_stories.json', 'w', encoding='utf-8') as f:
    json.dump({'user_stories': user_stories}, f, indent=2, ensure_ascii=False)
print("Saved: artifacts/user_stories.json")
print(str(user_stories)[:400])

## Phase 2: Construction
Generate system design (UML diagrams) and implementation code.

#### Generate UML Use Case Diagram

In [ ]:
prompt = f"""Generate a UML-compliant PlantUML use case diagram from these user stories.

Requirements:
- Define actors with the actor keyword, placed outside the system boundary
- Wrap use cases in rectangle "System" {{ ... }}
- Use -- for actor-to-usecase associations (not arrows)
- One use case per distinct goal, named with verb-noun phrases (max 5 words)
- Extract actors from the role field, use singular names
- Return ONLY valid PlantUML code, no explanations

User Stories: {user_stories}"""
puml_uc = get_completion(prompt, client, model_name, provider, temperature=0.3)
puml_uc = clean_llm_output(puml_uc, language='text')
print(puml_uc)

os.makedirs('artifacts/diagrams', exist_ok=True)
with open('artifacts/diagrams/use_case.puml', 'w') as f:
    f.write(puml_uc)
print("Saved: artifacts/diagrams/use_case.puml")

try:
    render_plantuml_diagram(puml_uc, "artifacts/diagrams/use_case.png")
except Exception as e:
    print(f"PlantUML render skipped: {e}")

#### Generate UML Class Diagram

In [ ]:
prompt = f"""Generate a PlantUML class diagram for a tourism recommendation system.
Include these entities: User, City, Attraction, TravelPlan, ItineraryItem.
Show relationships, attributes, and methods. Use standard UML notation.
Return ONLY valid PlantUML code, no explanations.

PRD Summary: {prd[:300]}"""
puml_class = get_completion(prompt, client, model_name, provider, temperature=0.3)
puml_class = clean_llm_output(puml_class, language='text')
print(puml_class)

with open('artifacts/diagrams/class_diagram.puml', 'w') as f:
    f.write(puml_class)
print("Saved: artifacts/diagrams/class_diagram.puml")

try:
    render_plantuml_diagram(puml_class, "artifacts/diagrams/class_diagram.png")
except Exception as e:
    print(f"PlantUML render skipped: {e}")

#### Generate Application Code

In [ ]:
prompt = """Create a Python dictionary called CITIES_DATA for a tourism app.
Include 6 Chinese cities (Beijing, Shanghai, Chengdu, Xi'an, Hangzhou, Guilin).
Each city has: name_cn, name_en, description (1 sentence),
attractions (list of 4-5 dicts, each with name, category, hours, ticket_price, highlight).
Output as a runnable Python dict assignment. No other code.
Format: CITIES_DATA = { ... }"""
cities_data = get_completion(prompt, client, model_name, provider, temperature=0.3)
cities_data = clean_llm_output(cities_data, language='python')
print("Data layer generated.")

os.makedirs('artifacts/app/flask', exist_ok=True)
with open('artifacts/app/flask/cities_data.py', 'w', encoding='utf-8') as f:
    f.write(cities_data)
print("Saved: artifacts/app/flask/cities_data.py")

In [ ]:
prompt = f"""Create a Flask API for a tourism recommendation system.
Import cities_data from cities_data module (from cities_data import CITIES_DATA).

Endpoints needed:
  GET /health -> {{status: ok}}
  GET /api/cities -> return list of all city names
  GET /api/attractions?city=<name> -> return city info + attractions list
  POST /api/plan -> body: {{city, days}}, return a day-by-day travel plan
  GET / -> serve index.html frontend

CRITICAL RULES:
- NO hardcoded city names or attractions in route functions
- All data lookups must use CITIES_DATA dictionary (dict.get, list comprehensions)
- Route functions only: parse request -> query CITIES_DATA -> return response
- Use flask_cors for CORS support
- In-memory plan storage using a list
- Bind to 0.0.0.0:5005 for Docker compatibility
- Include send_from_directory for static files
- No if/elif chains matching specific cities

User Stories: {user_stories}"""
main_py = get_completion(prompt, client, model_name, provider, temperature=0.3)
main_py = clean_llm_output(main_py, language='python')

with open('artifacts/app/flask/main.py', 'w', encoding='utf-8') as f:
    f.write(main_py)
print("Saved: artifacts/app/flask/main.py")
print("--- Preview ---")
print(main_py[:500])

In [ ]:
import re
import_lines = []
for line in main_py.split("\n"):
    stripped = line.strip()
    if stripped.startswith("import ") or stripped.startswith("from "):
        import_lines.append(stripped)
    elif stripped and not stripped.startswith("#"):
        break

prompt = f"""List the pip package names for these Python imports, one per line.
Only include packages not in the standard library. No version numbers.
Imports: {"; ".join(import_lines)}"""
requirements = get_completion(prompt, client, model_name, provider, temperature=0.2)
requirements = clean_llm_output(requirements, language="text")

req_lines = sorted(set(line.strip() for line in requirements.split("\n") if line.strip() and not line.startswith("#")))
clean_reqs = "\n".join(req_lines)

with open("artifacts/app/flask/requirements.txt", "w") as f:
    f.write(clean_reqs)
print(f"Saved: artifacts/app/flask/requirements.txt ({len(req_lines)} packages)")
print(clean_reqs)

#### Generate AI Images for Attractions

In [ ]:
# Generate a sample AI image for tourism (using APIFREE image model)
img_prompt = "A beautiful travel illustration of the Great Wall of China, colorful flat design style, tourist landmarks, vibrant and welcoming"
img_path, img_url = get_image_generation_completion(
    img_prompt, None, IMAGE_MODEL, "apifree"
)
if img_path and img_url:
    display(IPyImage(url=img_url))
    print(f"Sample image saved: {img_path}")
else:
    print(f"Image generation note: {img_url}")

#### Generate Web Frontend

In [ ]:
html_prompt = f"""Create a complete HTML5 webpage for a tourism recommendation system.

Requirements:
1. Bootstrap 5 via CDN for styling, clean modern look
2. Hero section with title 'AI Tourism Planner' and a subtitle
3. Search section: input for city name, select for number of days (1-7), Search button
4. Results section with two tabs: 'Attractions' and 'Travel Plan'
5. Attractions tab: cards showing each attraction with name, category, hours, ticket price
6. Travel Plan tab: day-by-day itinerary in an accordion layout
7. Image gallery: display generated images
8. All CSS in <style> tags, all JS in <script> tags
9. Use Fetch API: GET /api/attractions?city=X, POST /api/plan with {{city, days}}
10. Handle loading states, empty states, and network errors with user-friendly messages
11. Form validation: city is required, days must be 1-7
12. Responsive design, works on mobile
Output ONLY the complete HTML document. No markdown wrappers.
API Base URL: http://127.0.0.1:5005"""
html_code = get_completion(html_prompt, client, model_name, provider, temperature=0.4)
html_code = clean_llm_output(html_code, language="html")

with open("artifacts/app/flask/index.html", "w", encoding="utf-8") as f:
    f.write(html_code)
print("Saved: artifacts/app/flask/index.html")
print(f"HTML size: {len(html_code)} chars")

## Phase 3: Operation
Package the application for deployment and set up CI/CD pipeline.

#### Generate Docker Configuration

In [ ]:
# Generate Docker configuration files
dockerfile = """FROM python:3.10-slim
WORKDIR /app
COPY flask/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY flask/ .
EXPOSE 5005
ENV FLASK_APP=main.py
ENV FLASK_ENV=production
CMD ["python", "main.py"]
"""

dockerignore = """__pycache__
*.pyc
.env
.git
.pytest_cache
venv/
"""

docker_compose = """version: '3.8'
services:
  tourism-app:
    build:
      context: .
      dockerfile: docker/Dockerfile
    container_name: tourism-recommendation
    ports:
      - "5005:5005"
    environment:
      - FLASK_ENV=production
    volumes:
      - ./flask:/app
    restart: unless-stopped
"""

os.makedirs('artifacts/app/docker', exist_ok=True)
with open('artifacts/app/docker/Dockerfile', 'w') as f:
    f.write(dockerfile)
with open('artifacts/app/docker/.dockerignore', 'w') as f:
    f.write(dockerignore)
with open('artifacts/app/docker-compose.yml', 'w') as f:
    f.write(docker_compose)

print("Docker config files created.")
print()
print("Build & Run:")
print("  cd artifacts/app && docker-compose up --build")
print("  http://127.0.0.1:5005")

#### Generate CI/CD Pipeline (GitHub Actions)

In [ ]:
# Generate GitHub Actions workflow for CI/CD
github_actions_yaml = """name: CI/CD Pipeline
on:
  push:
    branches: [master, main]
  pull_request:
    branches: [master, main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      - name: Install dependencies
        run: |
          cd artifacts/app/flask
          pip install -r requirements.txt
      - name: Run tests
        run: |
          cd artifacts/app/flask
          python -m pytest test_api.py -v

  build-and-deploy:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Build Docker image
        run: |
          cd artifacts/app
          docker build -t tourism-app -f docker/Dockerfile .
      - name: Run container
        run: |
          cd artifacts/app
          docker-compose up -d
      - name: Health check
        run: |
          sleep 5
          curl -f http://localhost:5005/health || exit 1
"""

os.makedirs('.github/workflows', exist_ok=True)
with open('.github/workflows/deploy.yml', 'w') as f:
    f.write(github_actions_yaml)
print("Saved: .github/workflows/deploy.yml")
print("CI/CD pipeline configured: test → build → deploy → health check")

#### Generate API Tests

In [ ]:
prompt = f"""Write a pytest test file for a Flask tourism API. Use the Flask test client.

Test cases:
  1. test_health_check: GET /health returns 200 and {{status: ok}}
  2. test_get_cities: GET /api/cities returns list with at least 1 city
  3. test_get_attractions: GET /api/attractions?city=Beijing returns 200 with attractions
  4. test_get_attractions_invalid: GET /api/attractions?city=Atlantis returns 404
  5. test_create_plan: POST /api/plan with {{city: Beijing, days: 3}} returns 200
  6. test_create_plan_invalid: POST /api/plan with missing fields returns 400

Import the app from main module. Use pytest fixtures if needed.
Keep tests short, clear, with descriptive test function names.
Output ONLY the Python code, no explanations.

API Code: {main_py[:300]}"""
test_code = get_completion(prompt, client, model_name, provider, temperature=0.3)
test_code = clean_llm_output(test_code, language="python")

with open("artifacts/app/flask/test_api.py", "w", encoding="utf-8") as f:
    f.write(test_code)
print("Saved: artifacts/app/flask/test_api.py")
print(test_code[:500])

## Summary

The notebook has generated a complete AI-powered tourism recommendation system:

| Artifact | Output |
|----------|--------|
| SDLC Docs | PRD, Personas, User Stories |
| UML | Use Case Diagram, Class Diagram |
| Backend | Flask API (cities, attractions, travel plans) |
| Frontend | Bootstrap 5 responsive website |
| Images | AI-generated landmark images |
| Docker | Containerized deployment |
| CI/CD | GitHub Actions pipeline |
| Tests | pytest API test suite |

To deploy: `cd artifacts/app && docker-compose up --build`, then visit `http://127.0.0.1:5005`.